In [1]:
CLASS_MAP = {
    "fist": 0,

    "one": 1,
    "like": 1,
    "mute": 1,

    "peace": 2,
    "peace_inverted": 2,
    "two_up": 2,
    "two_up_inverted": 2,
    "rock": 2,
    "call": 2,

    "three": 3,
    "three2": 3,

    "four": 4,

    "palm": 5,
    "stop": 5,
    "stop_inverted": 5
}

In [2]:
import os
import json
import random
import shutil
from collections import defaultdict
from tqdm import tqdm

In [3]:
ROOT = "/kaggle/input/datasets/innominate817/hagrid-sample-30k-384p/hagrid-sample-30k-384p"

ANN_ROOT = f"{ROOT}/ann_train_val"
IMG_ROOT = f"{ROOT}/hagrid_30k"

OUT = "/kaggle/working/fingers_yolo"

In [4]:
image_index = {}

for root, _, files in os.walk(IMG_ROOT):
    for file in files:
        if file.endswith(".jpg"):
            image_id = file.replace(".jpg", "")
            image_index[image_id] = os.path.join(root, file)

print("Images:", len(image_index))

Images: 31833


In [5]:
samples_by_class = defaultdict(list)

for gesture in CLASS_MAP:

    json_path = os.path.join(
        ANN_ROOT,
        f"{gesture}.json"
    )

    with open(json_path) as f:
        annotations = json.load(f)

    target_class = CLASS_MAP[gesture]

    for image_id, item in annotations.items():

        if image_id not in image_index:
            continue

        bbox = item["bboxes"][0]

        samples_by_class[target_class].append({
            "image_id": image_id,
            "image_path": image_index[image_id],
            "bbox": bbox,
            "class": target_class,
            "user_id": item["user_id"]
        })

In [6]:
for cls in sorted(samples_by_class):
    print(cls, len(samples_by_class[cls]))

0 1735
1 5321
2 10630
3 3488
4 1805
5 5321


In [7]:
import random

TARGET_COUNTS = {
    0: 1735,
    1: 3500,
    2: 3500,
    3: 3488,
    4: 1805,
    5: 3500,
}

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

train = []
val = []
test = []

for cls in sorted(samples_by_class.keys()):

    samples = samples_by_class[cls]
    random.shuffle(samples)

    target = TARGET_COUNTS[cls]

    if len(samples) > target:
        samples = samples[:target]

    n = len(samples)

    train_end = int(n * TRAIN_RATIO)
    val_end = train_end + int(n * VAL_RATIO)

    train.extend(samples[:train_end])
    val.extend(samples[train_end:val_end])
    test.extend(samples[val_end:])

print("Train:", len(train))
print("Val:", len(val))
print("Test:", len(test))

Train: 12268
Val: 2628
Test: 2632


In [8]:
samples_by_class = defaultdict(list)

for gesture in CLASS_MAP.keys():

    json_path = os.path.join(ANN_ROOT, f"{gesture}.json")

    if not os.path.exists(json_path):
        continue

    with open(json_path, "r") as f:
        annotations = json.load(f)

    target_class = CLASS_MAP[gesture]

    for image_id, item in annotations.items():

        if image_id not in image_index:
            continue

        labels = item["labels"]
        bboxes = item["bboxes"]

        bbox = None

        for lbl, box in zip(labels, bboxes):
            if lbl == gesture:
                bbox = box
                break

        if bbox is None:
            continue

        samples_by_class[target_class].append({
            "image_id": image_id,
            "image_path": image_index[image_id],
            "bbox": bbox,
            "class": target_class
        })

In [9]:
TARGET_COUNTS = {
    0: 1735,
    1: 3500,
    2: 3500,
    3: 3488,
    4: 1805,
    5: 3500
}

In [10]:
balanced = defaultdict(list)

for cls, samples in samples_by_class.items():

    random.shuffle(samples)

    target = TARGET_COUNTS[cls]

    if len(samples) > target:
        samples = samples[:target]

    balanced[cls] = samples

In [11]:
train, val, test = [], [], []

for cls, samples in balanced.items():

    n = len(samples)

    train_end = int(n * 0.7)
    val_end = int(n * 0.85)

    train.extend(samples[:train_end])
    val.extend(samples[train_end:val_end])
    test.extend(samples[val_end:])

In [12]:
for split in ["train", "val", "test"]:
    os.makedirs(f"{OUT}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUT}/labels/{split}", exist_ok=True)

In [13]:
def export(data, split):

    for s in tqdm(data, desc=split):

        img_id = s["image_id"]

        shutil.copy2(
            s["image_path"],
            f"{OUT}/images/{split}/{img_id}.jpg"
        )

        x, y, w, h = s["bbox"]

        x_center = x + w / 2
        y_center = y + h / 2

        label_file = f"{OUT}/labels/{split}/{img_id}.txt"

        with open(label_file, "w") as f:
            f.write(
                f"{s['class']} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n"
            )

In [14]:
export(train, "train")
export(val, "val")
export(test, "test")

test: 100%|██████████| 2631/2631 [00:10<00:00, 242.41it/s]


In [15]:
yaml_text = """
path: /kaggle/working/hagrid_yolo

train: images/train
val: images/val
test: images/test

names:
  0: zero
  1: one
  2: two
  3: three
  4: four
  5: five
"""

with open(f"{OUT}/data.yaml", "w") as f:
    f.write(yaml_text)

In [16]:
from pathlib import Path

for split in ["train", "val", "test"]:
    imgs = len(list(Path(f"{OUT}/images/{split}").glob("*.jpg")))
    lbls = len(list(Path(f"{OUT}/labels/{split}").glob("*.txt")))

    print(split, imgs, lbls)

train 12267 12267
val 2629 2629
test 2631 2631
